In [8]:
from pathlib import Path
import sys

from scipy.optimize import linear_sum_assignment
import torch

In [9]:
x, target = torch.load("batch.pt", weights_only=False)
x.shape, target["boxes"].shape, target["labels"].shape, target["object_mask"].shape

(torch.Size([32, 3, 128, 128]),
 torch.Size([32, 3, 4]),
 torch.Size([32, 3]),
 torch.Size([32, 3]))

In [10]:
TORCH_SEED = 42
BATCH_SIZE = 32
NUM_CLASSES = 3  # background + rectangle + circle
MAX_NUM_OBJS = 100
IMAGE_SIZE = 128

torch.manual_seed(TORCH_SEED)

xy_min = torch.rand(BATCH_SIZE, MAX_NUM_OBJS, 2) * IMAGE_SIZE
xy_max = xy_min + torch.rand(BATCH_SIZE, MAX_NUM_OBJS, 2) * (IMAGE_SIZE - xy_min)
pred_locations = torch.cat([xy_min, xy_max], dim=-1)

pred_class_probs = torch.softmax(
    torch.randn(BATCH_SIZE, MAX_NUM_OBJS, NUM_CLASSES),
    dim=-1,
)

pred_locations.shape, pred_class_probs.shape

(torch.Size([32, 100, 4]), torch.Size([32, 100, 3]))

In [11]:
def hungarian_match(
    pred_locations,
    pred_class_probs,
    target,
    bbox_cost_weight=1.0,
    class_cost_weight=1.0,
):
    target_boxes = target["boxes"].to(pred_locations.device)
    target_labels = target["labels"].to(pred_locations.device)
    object_mask = target["object_mask"].to(pred_locations.device)

    # Compute pairwise costs against every padded target slot first.
    bbox_cost = torch.cdist(pred_locations, target_boxes, p=2)
    label_indices = target_labels.unsqueeze(1).expand(-1, pred_class_probs.shape[1], -1)
    class_cost = -pred_class_probs.gather(dim=2, index=label_indices)
    pairwise_cost = bbox_cost_weight * bbox_cost + class_cost_weight * class_cost

    # Only after the pairwise distances/costs exist, mask padded target slots.
    masked_pairwise_cost = pairwise_cost.masked_fill(
        ~object_mask.unsqueeze(1), torch.inf
    )

    matches = []
    for batch_idx in range(masked_pairwise_cost.shape[0]):
        valid_target_indices = object_mask[batch_idx].nonzero(as_tuple=True)[0]

        if valid_target_indices.numel() == 0:
            matches.append(
                (
                    torch.empty(0, dtype=torch.long),
                    torch.empty(0, dtype=torch.long),
                )
            )
            continue

        sample_cost = masked_pairwise_cost[batch_idx, :, valid_target_indices]
        pred_indices, valid_target_positions = linear_sum_assignment(
            sample_cost.cpu().numpy()
        )

        matches.append(
            (
                torch.as_tensor(pred_indices, dtype=torch.long),
                valid_target_indices[
                    torch.as_tensor(valid_target_positions, dtype=torch.long)
                ].cpu(),
            )
        )

    return matches, pairwise_cost, masked_pairwise_cost


matches, pairwise_cost, masked_pairwise_cost = hungarian_match(
    pred_locations=pred_locations,
    pred_class_probs=pred_class_probs,
    target=target,
)

matches[0], pairwise_cost.shape, masked_pairwise_cost.shape

((tensor([90]), tensor([0])),
 torch.Size([32, 100, 3]),
 torch.Size([32, 100, 3]))